# LoCED × MaxText v0.2.5 — Kaggle GPU validation

This notebook validates the **LoCED MaxText port** on a Kaggle GPU:

- creates a **Python 3.12** environment and installs **MaxText v0.2.5** with the CUDA 12 extras;
- checks that JAX sees the GPUs and uses the **real MaxText utilities** (not the CPU stubs);
- verifies the locked architecture: **4 physical layers, 8 visits, 4 FULL + REUSE/REUSE/REINDEX/REINDEX**;
- runs the **14 integration tests** (exact parity, MaxText Muon, sharding, checkpoint/resume);
- runs a short **MaxText Muon training benchmark**;
- verifies **Orbax checkpoint save → restore → resume** on the GPU.

### Kaggle checklist

1. **Accelerator → GPU** (prefer **T4 x2** or **L4**; newer JAX CUDA
   wheels target sm_70+). Internet must be **ON**.
2. Put the LoCED code on GitHub. This notebook clones `LOCED_REPO_URL` (default
   `https://github.com/offx-zinth/loced.git`) at `LOCED_REPO_REF`
   (default `master`). If your branch is different, edit those two values in the
   next cell.
3. Run the cells in order. The first install takes roughly 5–15 minutes.

### What “pass” means here

- Integration tests pass with `LOCED_REQUIRE_REAL_MAXTEXT_UTILS=1` — so MaxText's
  real mesh, LR schedule, Muon optimizer, and Orbax path were exercised.
- Parity uses `LOCED_PARITY_TOL=1e-5` on GPU. The default CPU run is exact
  (`0.0`); GPU fusion can introduce tiny last-bit differences.
- This is still the **tiny** config (549,456 parameters, sequence length 64).
  It validates correctness/plumbing on GPU, not large-scale throughput.

For TPU later: set `ACCELERATOR = "tpu"` in the configuration cell and switch the
Kaggle accelerator to TPU. The install uses `maxtext[tpu]`.

## 0. Configuration — repository, paths, accelerator

The notebook runs all heavy work in a separate Python 3.12 venv. That avoids
Kaggle's kernel Python version and keeps MaxText's pinned JAX stack isolated.

In [ ]:
# ============================================================================
# LoCED x MaxText v0.2.5 - Kaggle GPU validation
# ============================================================================
import json
import os
import shutil
import subprocess
import sys
import textwrap
import time
from pathlib import Path

# ---------------------------------------------------------------------------
# User knobs
# ---------------------------------------------------------------------------
ACCELERATOR = os.environ.get("LOCED_ACCELERATOR", "gpu")  # "gpu" now, "tpu" later
MAXTEXT_REF = "maxtext-v0.2.5"
MAXTEXT_REPO = "https://github.com/AI-Hypercomputer/maxtext.git"
MAXTEXT_EXTRAS = "cuda12" if ACCELERATOR == "gpu" else "tpu"

# The notebook clones this repo if LoCED is not already present.
LOCED_REPO_URL = os.environ.get(
    "LOCED_REPO_URL", "https://github.com/offx-zinth/loced.git"
)
LOCED_REPO_REF = os.environ.get("LOCED_REPO_REF", "master")

WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
WORK.mkdir(parents=True, exist_ok=True)
MAXTEXT_DIR = WORK / "maxtext-v0.2.5"
VENV_DIR = WORK / "loced-venv"
PY = VENV_DIR / "bin" / "python"
SCRIPTS = WORK / "loced_scripts"
RUNS = WORK / "loced_runs"
CACHE = WORK / "jax_cache"
for _p in (SCRIPTS, RUNS, CACHE):
    _p.mkdir(parents=True, exist_ok=True)


def _loced_parent(root: Path):
    # Return the directory to put on PYTHONPATH for `import loced`.
    if (root / "loced" / "maxtext" / "train.py").is_file():
        return root
    # offx-zinth/loced is the package contents at the repo root.
    if root.name == "loced" and (root / "maxtext" / "train.py").is_file():
        return root.parent
    return None


def _find_loced_root():
    candidates = []
    if os.environ.get("LOCED_ROOT"):
        candidates.append(Path(os.environ["LOCED_ROOT"]))
    candidates += [
        WORK,
        WORK / "loced",
        Path.cwd(),
        Path.cwd() / "loced",
        Path("/kaggle/input"),
    ]
    for root in candidates:
        found = _loced_parent(root)
        if found is not None:
            return found
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        for p in kaggle_input.glob("**/maxtext/train.py"):
            repo = p.parents[1]
            if repo.name == "loced" or (
                (repo / "attention.py").is_file()
                and (repo / "config.py").is_file()
            ):
                return repo.parent
        for p in kaggle_input.glob("**/loced/maxtext/train.py"):
            return p.parents[2]
    return None


# Keep a reused Kaggle working directory in sync with the branch under test.
_target = WORK / "loced"
if LOCED_REPO_URL and (_target / ".git").is_dir():
    _env = os.environ.copy()
    _env["GIT_TERMINAL_PROMPT"] = "0"
    print(f"updating existing checkout {_target} ({LOCED_REPO_REF})")
    subprocess.run(
        ["git", "-C", str(_target), "fetch", "--depth", "1",
         "origin", LOCED_REPO_REF],
        check=True, env=_env,
    )
    subprocess.run(
        ["git", "-C", str(_target), "reset", "--hard", "FETCH_HEAD"],
        check=True, env=_env,
    )

LOCED_ROOT = _find_loced_root()
if LOCED_ROOT is None and LOCED_REPO_URL:
    target = WORK / "loced"
    if not (target / ".git").is_dir():
        if target.is_dir() and any(target.iterdir()):
            print(f"{target} exists without .git; treating it as the source")
        else:
            env = os.environ.copy()
            env["GIT_TERMINAL_PROMPT"] = "0"
            print(f"cloning {LOCED_REPO_URL} ({LOCED_REPO_REF}) -> {target}")
            subprocess.run(
                ["git", "clone", "--depth", "1", "--branch", LOCED_REPO_REF,
                 LOCED_REPO_URL, str(target)],
                check=True, env=env,
            )
    LOCED_ROOT = _find_loced_root()

print(f"ACCELERATOR : {ACCELERATOR}")
print(f"WORK        : {WORK}")
print(f"LOCED_ROOT  : {LOCED_ROOT}")
print(f"MAXTEXT_DIR : {MAXTEXT_DIR}")
print(f"VENV        : {VENV_DIR}")
if LOCED_ROOT is None:
    raise RuntimeError(textwrap.dedent('''
        LoCED source not found.
        - Upload the repository folder (containing `loced/`) to
          /kaggle/working/loced, or as a Kaggle Dataset under
          /kaggle/input/...
        - Or set LOCED_REPO_URL (and LOCED_REPO_REF) in this cell to a branch
          that contains `loced/`.
    '''))
print("nvidia-smi:")
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi", "-L"], check=False)
else:
    print("nvidia-smi not found (CPU-only notebook kernel?)")

## 1. Python 3.12 + MaxText v0.2.5 (CUDA 12)

MaxText v0.2.5 requires Python >= 3.12, so the notebook creates its own venv
with `uv` even if the Kaggle kernel is older. The install uses MaxText's tested
dependency set:

```bash
uv pip install -e .[cuda12] --resolution=lowest
```

Set `LOCED_SKIP_INSTALL=1` only when re-running against an already prepared
venv.

In [ ]:
SKIP_INSTALL = os.environ.get("LOCED_SKIP_INSTALL", "0") == "1"

if SKIP_INSTALL:
    print("LOCED_SKIP_INSTALL=1: using existing venv/MaxText checkout")
else:
    if shutil.which("uv") is None:
        print("installing uv ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    UV = shutil.which("uv")
    if UV is None:
        raise RuntimeError("uv is not on PATH after pip install")

    if not PY.exists():
        print("creating Python 3.12 venv ...")
        subprocess.run(
            [UV, "venv", "--python", "3.12", "--seed", str(VENV_DIR)],
            check=True,
        )
    print(subprocess.check_output([str(PY), "--version"], text=True).strip())

    if not (MAXTEXT_DIR / "pyproject.toml").is_file():
        print(f"cloning MaxText {MAXTEXT_REF} ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", MAXTEXT_REF,
             MAXTEXT_REPO, str(MAXTEXT_DIR)],
            check=True,
        )
    else:
        print("MaxText checkout already present")

    print(f"installing maxtext[{MAXTEXT_EXTRAS}] (resolution=lowest) ...")
    subprocess.run(
        [UV, "pip", "install", "--python", str(PY), "--link-mode=copy",
         "-e", f".[{MAXTEXT_EXTRAS}]", "--resolution=lowest"],
        cwd=str(MAXTEXT_DIR), check=True,
    )
    subprocess.run(
        [UV, "pip", "install", "--python", str(PY), "--link-mode=copy",
         "pytest", "pytest-timeout"],
        check=True,
    )
    print("dependency check:")
    subprocess.run([UV, "pip", "check", "--python", str(PY)], check=False)

print(subprocess.check_output(
    [str(PY), "-c",
     "import sys, maxtext; print(sys.version); print(maxtext.__file__)"],
    text=True,
))

## 2. GPU + real-MaxText sanity check

This is deliberately strict: the run fails if JAX is not on the GPU, or if
MaxText's heavy utilities cannot be imported. A CPU-stub fallback would make the
rest of the notebook meaningless on Kaggle.

In [ ]:
BASE_ENV = os.environ.copy()
BASE_ENV.update({
    "PYTHONPATH": f"{MAXTEXT_DIR / 'src'}:{LOCED_ROOT}",
    "JAX_COMPILATION_CACHE_DIR": str(CACHE),
    "XLA_PYTHON_CLIENT_MEM_FRACTION": "0.90",
    "XLA_PYTHON_CLIENT_PREALLOCATE": "true",
    "TOKENIZERS_PARALLELISM": "false",
    "HF_HUB_DISABLE_PROGRESS_BARS": "1",
    "PYTHONDONTWRITEBYTECODE": "1",
    "TF_CPP_MIN_LOG_LEVEL": "2",
})

# jax-cuda12 loads the pip-installed NVIDIA libraries with ctypes.  In a venv
# they are under site-packages/nvidia/*/lib, which is not on the default loader
# path, so cuSPARSE/cuDNN can otherwise be reported as "not found".
_nvidia_code = (
    "import site; from pathlib import Path; "
    "print(':'.join(str(p) for r in site.getsitepackages() "
    "for p in sorted((Path(r) / 'nvidia').glob('*/lib')) if p.is_dir()))"
)
_nvidia_libs = subprocess.check_output([str(PY), "-c", _nvidia_code], text=True).strip()
if _nvidia_libs:
    BASE_ENV["LD_LIBRARY_PATH"] = ":".join(
        x for x in (_nvidia_libs, os.environ.get("LD_LIBRARY_PATH", "")) if x
    )
print("NVIDIA lib dirs:", len(_nvidia_libs.split(":")) if _nvidia_libs else 0)


def run_script(name, code, timeout=3600, extra_env=None, check=True, cwd=None):
    path = SCRIPTS / name
    path.write_text(textwrap.dedent(code))
    env = BASE_ENV.copy()
    if extra_env:
        env.update({str(k): str(v) for k, v in extra_env.items()})
    t0 = time.time()
    proc = subprocess.run(
        [str(PY), str(path)],
        cwd=str(cwd or LOCED_ROOT),
        env=env,
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    elapsed = time.time() - t0
    print(proc.stdout)
    if proc.stderr.strip():
        print("[stderr]\n" + proc.stderr[-12000:])
    print(f"[{name}] exit={proc.returncode} elapsed={elapsed:.1f}s")
    if check and proc.returncode != 0:
        raise RuntimeError(f"{name} failed (exit {proc.returncode})")
    return proc


gpu_check = r'''
import importlib.metadata
import os
import shutil
import subprocess
import sys

import jax
import maxtext

print("python  :", sys.version.split()[0])
print("jax     :", jax.__version__)
print("maxtext :", importlib.metadata.version("maxtext"))
print("backend :", jax.default_backend())
print("devices :", jax.devices())

expected = os.environ["LOCED_EXPECTED_BACKEND"]
assert jax.default_backend() == expected, (
    f"Expected {expected}, got {jax.default_backend()}. "
    "Set the Kaggle accelerator and restart the session."
)
assert len(jax.devices()) >= 1
for d in jax.devices():
    stats = d.memory_stats() or {}
    print(f"  {d}: {d.device_kind} bytes_limit={stats.get('bytes_limit')}")

# The GPU/TPU run must exercise the real MaxText utilities, not CPU stubs.
import maxtext.utils.maxtext_utils  # noqa: F401
import maxtext.utils.model_creation_utils  # noqa: F401
print("real MaxText utilities: OK")
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
'''
run_script(
    "gpu_check.py", gpu_check,
    extra_env={"LOCED_EXPECTED_BACKEND": ACCELERATOR},
)

## 3. Architecture + sharding on the GPU

Builds the NNX model with MaxText's real mesh and logical-axis rules. This cell
fails if the 4-layer / 8-visit schedule or the compressed-attention fields have
drifted.

In [ ]:
arch_code = r'''
import json
import os

import jax
from maxtext.configs import pyconfig

from loced.maxtext.config import LoCEDMaxTextConfig, loced_config_from_maxtext
from loced.maxtext.train import create_model_and_mesh

accelerator = os.environ["LOCED_ACCELERATOR"]
n = len(jax.devices())
if accelerator == "gpu":
    mesh_overrides = [f"ici_fsdp_parallelism={n}", "ici_data_parallelism=1"]
else:
    mesh_overrides = ["ici_fsdp_parallelism=1", f"ici_data_parallelism={n}"]

config = pyconfig.initialize(
    ["loced-arch", "loced/maxtext/configs/loced_tiny.yml", *mesh_overrides,
     "steps=2", "per_device_batch_size=2", "enable_checkpointing=false",
     "log_config=false", "skip_jax_distributed_system=true"],
    config_class=LoCEDMaxTextConfig,
)
cfg = loced_config_from_maxtext(config)
model, mesh = create_model_and_mesh(config)

print("params           :", f"{model.count_params():,}")
print("physical layers  :", cfg.num_layers)
print("visits           :", cfg.num_visits)
print("encode           :", cfg.encode_layers)
print("decode           :", cfg.decode_layers)
print("pool/window/top_k:", cfg.pool_m, cfg.window, cfg.top_k)
print("indexer          :", cfg.indexer_heads, "heads x", cfg.indexer_dim)
print("experts          :", cfg.num_experts, "top", cfg.top_k_experts,
      "shared", cfg.num_shared_experts)
print("mesh axes        :", mesh.axis_names)
print("mesh shape       :", mesh.shape)

assert cfg.num_layers == 4
assert cfg.num_visits == 8
assert cfg.encode_layers == ("full",) * 4
assert cfg.decode_layers == ("full",) * 4 + (
    "reuse", "reuse", "reindex", "reindex")
assert cfg.pool_m >= 1 and cfg.window >= 0 and cfg.top_k >= 0

for name, param in [
    ("token_embedding", model.token_embedding),
    ("layer0.W_q", model.cells["0"].attn.W_q),
    ("layer0.experts.wi_0", model.cells["0"].moe.routed_moe.wi_0),
    ("layer0.norm_a", model.cells["0"].norm_a),
]:
    print(f"{name:22s}", param.get_value().sharding)

report = {
    "backend": jax.default_backend(),
    "devices": n,
    "params": model.count_params(),
    "num_layers": cfg.num_layers,
    "num_visits": cfg.num_visits,
    "encode": list(cfg.encode_layers),
    "decode": list(cfg.decode_layers),
    "pool_m": cfg.pool_m,
    "window": cfg.window,
    "top_k": cfg.top_k,
    "indexer_heads": cfg.indexer_heads,
    "indexer_dim": cfg.indexer_dim,
    "num_experts": cfg.num_experts,
    "top_k_experts": cfg.top_k_experts,
}
with open(os.environ["LOCED_ARCH_REPORT"], "w") as f:
    json.dump(report, f, indent=2)
print("architecture report written")
'''
run_script(
    "architecture.py", arch_code,
    extra_env={
        "LOCED_ACCELERATOR": ACCELERATOR,
        "LOCED_ARCH_REPORT": str(WORK / "arch_report.json"),
    },
)

## 4. Full MaxText integration suite

Runs `loced/maxtext/test_integration.py` with:

- `LOCED_REQUIRE_REAL_MAXTEXT_UTILS=1` — no silent CPU stubs;
- `LOCED_PARITY_TOL=1e-5` — GPU parity tolerance (CPU default is exact `0.0`).

This covers weights, loss/logits, gradients, prefill/decode, MaxText Muon,
sharding classification, config, checkpoint round-trip, and trainer resume.

In [ ]:
test_env = BASE_ENV.copy()
test_env.update({
    "LOCED_PARITY_TOL": "1e-5",
    "LOCED_REQUIRE_REAL_MAXTEXT_UTILS": "1",
    "LOCED_ACCELERATOR": ACCELERATOR,
})
cmd = [
    str(PY), "-m", "pytest", "-q",
    "loced/maxtext/test_integration.py",
    "-x", "--durations=10", "--timeout=1800", "-p", "no:cacheprovider",
]
t0 = time.time()
proc = subprocess.run(
    cmd, cwd=str(LOCED_ROOT), env=test_env,
    text=True, capture_output=True, timeout=5400,
)
print(proc.stdout)
if proc.stderr.strip():
    print("[stderr]\n" + proc.stderr[-12000:])
print(f"pytest exit={proc.returncode} elapsed={time.time() - t0:.1f}s")
if proc.returncode != 0:
    raise RuntimeError("MaxText integration tests failed")
print("MaxText integration tests: PASS")

## 5. MaxText Muon training benchmark

Two warmup steps compile the jitted train step; the next 20 steps are timed.
This is a **global synthetic batch** on the tiny config, so read it as a
plumbing/throughput sanity check, not a model-scale benchmark.

In [ ]:
bench_code = r'''
import json
import os
import time

import jax
import numpy as np
from maxtext.configs import pyconfig

from loced.maxtext.config import LoCEDMaxTextConfig, loced_config_from_maxtext
from loced.maxtext.train import (
    create_model_and_mesh, create_optimizer, synthetic_batches, train_step,
)

accelerator = os.environ["LOCED_ACCELERATOR"]
n = len(jax.devices())
if accelerator == "gpu":
    mesh_overrides = [f"ici_fsdp_parallelism={n}", "ici_data_parallelism=1"]
else:
    mesh_overrides = ["ici_fsdp_parallelism=1", f"ici_data_parallelism={n}"]

warmup = int(os.environ.get("LOCED_BENCH_WARMUP", "2"))
steps = int(os.environ.get("LOCED_BENCH_STEPS", "20"))
batch = int(os.environ.get("LOCED_BENCH_BATCH", "4"))
config = pyconfig.initialize(
    ["loced-bench", "loced/maxtext/configs/loced_tiny.yml", *mesh_overrides,
     f"steps={warmup + steps}", f"per_device_batch_size={batch}",
     "enable_checkpointing=false", "log_config=false",
     "skip_jax_distributed_system=true"],
    config_class=LoCEDMaxTextConfig,
)
cfg = loced_config_from_maxtext(config)
model, mesh = create_model_and_mesh(config)
schedule, optimizer = create_optimizer(config, model, mesh)
print("params", f"{model.count_params():,}", "devices", n,
      "batch", batch, "seq", cfg.max_seq_len)

batches = list(synthetic_batches(
    cfg.vocab_size, cfg.max_seq_len, batch, warmup + steps, seed=0))
before = np.asarray(model.cells["0"].attn.W_q.get_value()).copy()

for i in range(warmup):
    loss, _ = train_step(model, optimizer, batches[i])
    jax.block_until_ready(loss)
print("warmup done")

losses = []
t0 = time.perf_counter()
for i in range(warmup, warmup + steps):
    loss, metrics = train_step(model, optimizer, batches[i])
    losses.append(float(loss))
    jax.block_until_ready(loss)
t1 = time.perf_counter()

elapsed = t1 - t0
tokens = steps * batch * cfg.max_seq_len
after = np.asarray(model.cells["0"].attn.W_q.get_value())
print("loss first/last:", losses[0], losses[-1])
print("loss delta     :", losses[-1] - losses[0])
print("optimizer step :", int(optimizer.step.get_value()))
print(f"step time      : {elapsed / steps * 1000:.1f} ms")
print(f"tokens/sec     : {tokens / elapsed:,.0f} (global synthetic batch)")
print("param changed  :", not np.array_equal(before, after))
for d in jax.devices():
    stats = d.memory_stats() or {}
    print(f"memory {d}: peak={stats.get('peak_bytes_in_use')} "
          f"limit={stats.get('bytes_limit')}")

assert np.isfinite(losses).all()
assert int(optimizer.step.get_value()) == warmup + steps
assert not np.array_equal(before, after)

report = {
    "backend": jax.default_backend(),
    "devices": n,
    "batch": batch,
    "seq_len": cfg.max_seq_len,
    "params": model.count_params(),
    "warmup_steps": warmup,
    "timed_steps": steps,
    "loss_first": losses[0],
    "loss_last": losses[-1],
    "ms_per_step": elapsed / steps * 1000,
    "tokens_per_sec": tokens / elapsed,
    "optimizer_step": int(optimizer.step.get_value()),
    "param_changed": bool(not np.array_equal(before, after)),
}
with open(os.environ["LOCED_TRAIN_REPORT"], "w") as f:
    json.dump(report, f, indent=2)
'''
run_script(
    "benchmark.py", bench_code, timeout=3600,
    extra_env={
        "LOCED_ACCELERATOR": ACCELERATOR,
        "LOCED_TRAIN_REPORT": str(WORK / "train_report.json"),
    },
)

## 6. Orbax checkpoint save → restore → resume

Checks that the GPU run can save model + optimizer state, restore it into a fresh
model, and continue from the saved optimizer step.

In [ ]:
ckpt_code = r'''
import json
import os
import shutil
from pathlib import Path

import jax
import numpy as np
from flax import nnx
from maxtext.configs import pyconfig

from loced.maxtext.config import LoCEDMaxTextConfig, loced_config_from_maxtext
from loced.maxtext.train import (
    checkpoint_path, create_model_and_mesh, create_optimizer,
    load_checkpoint, save_checkpoint, synthetic_batches, train_step,
)

accelerator = os.environ["LOCED_ACCELERATOR"]
n = len(jax.devices())
if accelerator == "gpu":
    mesh_overrides = [f"ici_fsdp_parallelism={n}", "ici_data_parallelism=1"]
else:
    mesh_overrides = ["ici_fsdp_parallelism=1", f"ici_data_parallelism={n}"]

base = os.environ["LOCED_CKPT_BASE"]
config = pyconfig.initialize(
    ["loced-ckpt", "loced/maxtext/configs/loced_tiny.yml", *mesh_overrides,
     "steps=3", "per_device_batch_size=2", "enable_checkpointing=true",
     "checkpoint_period=1", f"base_output_directory={base}",
     "run_name=loced-kaggle-ckpt", "log_config=false",
     "skip_jax_distributed_system=true"],
    config_class=LoCEDMaxTextConfig,
)
cfg = loced_config_from_maxtext(config)
ckpt = checkpoint_path(config)
shutil.rmtree(ckpt, ignore_errors=True)
print("checkpoint dir:", ckpt)

model, mesh = create_model_and_mesh(config)
schedule, optimizer = create_optimizer(config, model, mesh)
tokens = next(iter(synthetic_batches(
    cfg.vocab_size, cfg.max_seq_len, 2, 1, seed=0)))
for _ in range(2):
    loss, _ = train_step(model, optimizer, tokens)
    jax.block_until_ready(loss)
assert int(optimizer.step.get_value()) == 2
save_checkpoint(ckpt, model, optimizer)
print("saved at step", int(optimizer.step.get_value()))

model2, mesh2 = create_model_and_mesh(config)
_, optimizer2 = create_optimizer(config, model2, mesh2)
restored = load_checkpoint(ckpt, model2, optimizer2, mesh2)
print("restored step:", restored)
assert restored == 2
assert int(optimizer2.step.get_value()) == 2

a = jax.tree.leaves(nnx.to_pure_dict(nnx.state(model)))
b = jax.tree.leaves(nnx.to_pure_dict(nnx.state(model2)))
assert all(np.array_equal(np.asarray(x), np.asarray(y))
           for x, y in zip(a, b))

loss, _ = train_step(model2, optimizer2, tokens)
jax.block_until_ready(loss)
assert int(optimizer2.step.get_value()) == 3
print("resumed to step", int(optimizer2.step.get_value()))

files = [p for p in Path(ckpt).rglob("*") if p.is_file()]
size = sum(p.stat().st_size for p in files)
print("checkpoint files:", len(files), "bytes:", size)

report = {
    "checkpoint_dir": ckpt,
    "restored_step": restored,
    "resumed_step": int(optimizer2.step.get_value()),
    "files": len(files),
    "bytes": size,
}
with open(os.environ["LOCED_CKPT_REPORT"], "w") as f:
    json.dump(report, f, indent=2)
'''
run_script(
    "checkpoint.py", ckpt_code, timeout=3600,
    extra_env={
        "LOCED_ACCELERATOR": ACCELERATOR,
        "LOCED_CKPT_BASE": str(RUNS),
        "LOCED_CKPT_REPORT": str(WORK / "ckpt_report.json"),
    },
)

## 7. Summary + TPU handoff

If every cell above completed, the GPU run is green. For TPU:

1. Change the Kaggle accelerator to a TPU VM.
2. Set `ACCELERATOR = "tpu"` in the configuration cell.
3. Re-run from the install cell; it installs `maxtext[tpu]` and uses
   `ici_data_parallelism=<num_devices>`.
4. Keep the same tests/benchmark; the parity tolerance can be tightened or left
   at `1e-5` until TPU numerics are characterized.

In [ ]:
reports = {}
for name in ("arch_report.json", "train_report.json", "ckpt_report.json"):
    p = WORK / name
    if p.is_file():
        reports[name] = json.loads(p.read_text())

print(json.dumps(reports, indent=2))

summary = {
    "accelerator": ACCELERATOR,
    "backend": reports.get("arch_report.json", {}).get("backend"),
    "devices": reports.get("arch_report.json", {}).get("devices"),
    "params": reports.get("arch_report.json", {}).get("params"),
    "integration_tests": "PASS",
    "parity_tolerance": 1e-5,
    "train": reports.get("train_report.json"),
    "checkpoint": reports.get("ckpt_report.json"),
}
report_path = WORK / "kaggle_gpu_report.json"
report_path.write_text(json.dumps(summary, indent=2))
print("Wrote", report_path)
print("GPU validation complete. Next: set ACCELERATOR='tpu' and use a TPU VM.")